# 🛍️ ShopTalk — Phase 11 prep: Docker files, artifacts and GitHub

This notebook prepares everything Phase 11 needs **before** any AWS money is spent. It does not deploy anything.

| § | What |
|---|---|
| 1 | Setup: Drive, config, the same pinned packages as Phases 7–10 |
| 2 | Write `Dockerfile`, `.dockerignore`, `docker/start.sh`, `docker/smoke_test.py`, `deploy/ec2.sh`, `deploy/README.md` into the repo — then test the container's start script here in fake mode |
| 3 | `requirements-docker.txt`: exact versions from this (tested) Colab runtime |
| 4 | `shoptalk_artifacts.zip`: the LoRA embedding model + Chroma index (too big for GitHub; goes to S3) |
| 5 | Push the current Drive code (Phases 7–10) + the new files to GitHub |
| 6 | Checklist and next steps |

After this: follow **`deploy/README.md`** (local Docker test → EC2 → demo → shutdown).
Do the **GPU quota request today** (README step 1) — AWS approval can take a day or more.

## 1. Setup

### 📘 1.1 GPU not needed · mount Google Drive

**Purpose:** Mount Drive (with the fallbacks from Phase 10). A CPU runtime is fine for this notebook.

**Expected Results:** `✅ Drive mounted`.

**Parameters:** none

In [ ]:
import os
from google.colab import drive
MYDRIVE = "/content/drive/MyDrive"
if not os.path.isdir(MYDRIVE):
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print("drive.mount failed:", e, "→ trying auth.authenticate_user() first")
        from google.colab import auth
        auth.authenticate_user()
        drive.mount("/content/drive", force_remount=True)
assert os.path.isdir(MYDRIVE), "Drive not mounted — try the Files sidebar ▸ Mount Drive, then re-run"
print("✅ Drive mounted")

### 📘 1.2 Configuration (single config cell)

**Purpose:** Paths, GitHub repo and what to sync.

**Expected Results:** No output.

**Parameters:** `GITHUB_REPO`, `GIT_BRANCH`, `GIT_USER_NAME`, `GIT_USER_EMAIL`, `NOTEBOOK_GLOBS`, `PUSH`.

In [ ]:
# ===================== Phase 11 prep configuration =====================
from pathlib import Path
DRIVE_PROJECT  = Path("/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk")
REPO_LINK      = Path("/content/ShopTalk")                 # symlink (no spaces → safe for %%writefile)
RAG_CONFIG_REL = "src/rag/rag_config.json"
ARTIFACTS_DIR  = REPO_LINK / "deploy_artifacts"            # on Drive; download the zip from here for S3

# --- GitHub (the repo you already have)
GITHUB_REPO    = "https://github.com/<your-user>/<your-repo>.git"   # ← edit
GIT_BRANCH     = "main"
GIT_USER_NAME  = "Your Name"                                          # ← edit (commit author)
GIT_USER_EMAIL = "you@example.com"                                    # ← edit
PUSH           = False      # first run: False = show what would change; then True to commit and push

# --- What goes to GitHub (code + deployment files; no data, models, logs or results)
SYNC_DIRS   = ["src", "docker", "deploy"]
SYNC_FILES  = ["Dockerfile", ".dockerignore", "requirements-api.txt", "requirements-ui.txt", "requirements-docker.txt"]
# phase notebooks (07–11) — every match is copied to notebooks/ in the repo
NOTEBOOK_GLOBS = ["/content/drive/MyDrive/Colab Notebooks/*ShopTalk*.ipynb",
                  "/content/drive/MyDrive/Colab Notebooks/*Phase*.ipynb",
                  str(DRIVE_PROJECT / "notebooks" / "*.ipynb")]
GITIGNORE   = ["logs/", "results/", "data/", "models/", "deploy_artifacts/", "*.pem", "__pycache__/",
               "src/_backup_before_phase10/", ".ipynb_checkpoints/"]

### 📘 1.3 Install the pinned packages (same as Phases 7–10)

**Purpose:** Install `requirements-api.txt` (+ `requirements-ui.txt`) exactly as before (no `-U`), so section 3 records the versions the project was actually tested with.

**Expected Results:** The pins print ✅.

**Parameters:** none

In [ ]:
import os, sys, subprocess, shutil
from importlib.metadata import version, PackageNotFoundError
if not REPO_LINK.exists():
    REPO_LINK.symlink_to(DRIVE_PROJECT, target_is_directory=True)
for name in ("requirements-api.txt", "requirements-ui.txt"):
    p = REPO_LINK / name
    if p.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(p)], check=True)
        print("installed", name)
for pkg in ("torch", "transformers", "sentence-transformers", "peft", "chromadb", "torchao", "fastapi", "gradio", "python-multipart"):
    try: print(f"   {pkg:22s} {version(pkg)}")
    except PackageNotFoundError: print(f"❌ {pkg:22s} MISSING")

### 📘 1.4 Helpers, CHECKS and a pre-flight of the repo

**Purpose:** Confirm the repo on Drive has the Phase 10 code and the settings the deployment relies on.

**Expected Results:** `repo_ready` = PASS: `src/asr`, `src/tts`, the `asr` block and `max_new_tokens` in rag_config, the embedding model and Chroma index present.

**Parameters:** none

In [ ]:
import json, re, time, zipfile, hashlib, glob
import pandas as pd
CHECKS = {}
_ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "PENDING": "⏳", "SKIP": "⏭️"}
def check(name, status, detail=""):
    CHECKS[name] = {"status": status, "detail": str(detail)}
    print(f"{_ICON.get(status, '•')} {name}: {status} — {detail}")

RAG_CFG = json.loads((REPO_LINK / RAG_CONFIG_REL).read_text())
EMBED_DIR = REPO_LINK / RAG_CFG["embed_model"]
CHROMA_DIR = REPO_LINK / RAG_CFG["chroma_dir"]
problems = []
for rel in ("src/asr/whisper_asr.py", "src/tts/mms_tts.py", "src/api/main.py", "src/ui/app.py", "src/rag/factory.py"):
    if not (REPO_LINK / rel).exists(): problems.append(f"missing {rel}")
if "asr" not in RAG_CFG: problems.append("rag_config has no 'asr' block (run Phase 10 §4.4)")
if not EMBED_DIR.exists(): problems.append(f"embedding model not found: {RAG_CFG['embed_model']}")
if not (CHROMA_DIR / "chroma.sqlite3").exists(): problems.append(f"Chroma index not found: {RAG_CFG['chroma_dir']}")
print("LLM:", RAG_CFG["llm"]["model_id"], "| max_new_tokens:", RAG_CFG["llm"].get("max_new_tokens"),
      "| ASR:", RAG_CFG.get("asr", {}).get("model_id"), "| reranker:", RAG_CFG["retrieval"].get("reranker"))
check("repo_ready", "PASS" if not problems else "FAIL", "; ".join(problems) or "Phase 10 code, config and artifacts present")

## 2. Deployment files (`%%writefile` into the repo)

### 📘 2.1 `Dockerfile`

**Purpose:** one image for API + UI; torch from the CUDA 13.0 index by default (or the CPU index for the laptop test); models mounted, not baked in

**Expected Results:** `Writing /content/ShopTalk/Dockerfile`.

**Parameters:** none

In [ ]:
for d in ("docker", "deploy"): (REPO_LINK / d).mkdir(exist_ok=True)
print("ok")

In [ ]:
%%writefile /content/ShopTalk/Dockerfile
# ShopTalk — API (FastAPI) + UI (Gradio) in one image. Phase 11.
#
# GPU (EC2 g4dn, default):   docker build -t shoptalk:gpu .
# CPU (laptop, fake mode):   docker build --build-arg TORCH_INDEX=https://download.pytorch.org/whl/cpu -t shoptalk:cpu .
#
# Models are NOT baked in: the LoRA embedding model and the Chroma index are mounted at /app/models and
# /app/data, and Hugging Face downloads (Qwen, Whisper, reranker, MMS voice) are cached in a volume at
# /cache/huggingface, so the image stays small and a restart does not re-download anything.
ARG PYTHON_VERSION=3.13
FROM python:${PYTHON_VERSION}-slim

# Same torch build as the Colab runtime the project was tested on (2.14.1 + CUDA 13.0).
ARG TORCH_VERSION=2.14.1
ARG TORCH_INDEX=https://download.pytorch.org/whl/cu130

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1 \
    HF_HOME=/cache/huggingface \
    SHOPTALK_REPO_DIR=/app \
    SHOPTALK_LOG_DIR=/app/logs \
    SHOPTALK_LOCAL_CHROMA=/tmp/shoptalk_chroma \
    SHOPTALK_API_URL=http://127.0.0.1:8000 \
    SHOPTALK_UI_PORT=7860

# ffmpeg decodes browser recordings (Phase 10); tini forwards stop signals to both processes.
RUN apt-get update \
 && apt-get install -y --no-install-recommends ffmpeg tini \
 && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# 1) torch first, from the PyTorch index for the chosen device (biggest layer, changes least)
RUN pip install "torch==${TORCH_VERSION}" --index-url "${TORCH_INDEX}"

# 2) everything else, pinned to the versions tested in Colab (generated by the Phase 11 prep notebook)
COPY requirements-docker.txt .
RUN pip install -r requirements-docker.txt

# 3) application code last (changes most often)
COPY src/ src/
COPY docker/ docker/
RUN chmod +x docker/start.sh && mkdir -p /app/logs /app/models /app/data /cache/huggingface

EXPOSE 7860 8000
HEALTHCHECK --interval=30s --timeout=10s --start-period=600s --retries=3 \
  CMD python -c "import json,urllib.request,sys; h=json.load(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=5)); sys.exit(0 if h.get('status')=='ok' else 1)"

ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["/app/docker/start.sh"]

### 📘 2.2 `.dockerignore`

**Purpose:** allow-list: only `src/`, `docker/` and `requirements-docker.txt` go into the build

**Expected Results:** `Writing /content/ShopTalk/.dockerignore`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/.dockerignore
# Allow-list: only what the image needs. Models, data, logs and results are mounted at run time.
*
!Dockerfile
!requirements-docker.txt
!src/
!docker/
src/_backup_before_phase10/
**/__pycache__/
**/*.pyc

### 📘 2.3 `docker/start.sh`

**Purpose:** starts the API (8000) and the UI (7860); if either exits, the container exits so Docker can restart it

**Expected Results:** `Writing /content/ShopTalk/docker/start.sh`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/docker/start.sh
#!/usr/bin/env bash
# Starts the ShopTalk API (port 8000) and UI (port 7860) in one container.
# If either process exits, the container exits too, so Docker's restart policy can bring it back.
set -euo pipefail
cd "${APP_DIR:-/app}"

python -m uvicorn src.api.main:app --host 0.0.0.0 --port "${SHOPTALK_API_PORT:-8000}" --log-level warning &
API_PID=$!
python -m src.ui.app &
UI_PID=$!

stop() { kill -TERM "$API_PID" "$UI_PID" 2>/dev/null || true; wait || true; }
trap stop TERM INT

echo "[start.sh] API pid $API_PID on :${SHOPTALK_API_PORT:-8000} · UI pid $UI_PID on :${SHOPTALK_UI_PORT:-7860}"
echo "[start.sh] models load in the background; /health reports 'loading' until they are ready (first start downloads ~8 GB)"
set +e
wait -n "$API_PID" "$UI_PID"
code=$?
echo "[start.sh] a process exited with code $code — stopping the container"
stop
exit "$code"

### 📘 2.4 `docker/smoke_test.py`

**Purpose:** standard-library end-to-end check: health, recommend, follow-up, feedback, click, transcribe, recommend/audio, speak, metrics, optional latency sample

**Expected Results:** `Writing /content/ShopTalk/docker/smoke_test.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/docker/smoke_test.py
"""ShopTalk deployment smoke test (Phase 11). Standard library only, so it runs anywhere:

    docker exec shoptalk python /app/docker/smoke_test.py                # inside the container
    python docker/smoke_test.py --url http://localhost:8000              # from the host
    python docker/smoke_test.py --latency 30                             # + P50/P95 over 30 requests
    python docker/smoke_test.py --audio my_query.wav                     # use a real recording for /transcribe

Checks: /health (waits while models load), /recommend, a follow-up with history, /transcribe and
/recommend/audio, /speak (if enabled), /feedback, /click, /metrics. Exit code 0 = all passed.
Sessions are tagged "phase11-smoke" so they can be filtered out of log mining.
"""
from __future__ import annotations

import argparse
import io
import json
import math
import statistics
import struct
import sys
import time
import urllib.error
import urllib.request
import uuid
import wave

SESSION = "phase11-smoke"
RESULTS = []


def _req(url, method="GET", data=None, headers=None, timeout=180):
    req = urllib.request.Request(url, data=data, method=method, headers=headers or {})
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body, status, ctype = r.read(), r.status, r.headers.get("Content-Type", "")
    except urllib.error.HTTPError as e:
        body, status, ctype = e.read(), e.code, e.headers.get("Content-Type", "")
    ms = (time.perf_counter() - t0) * 1000
    payload = json.loads(body) if ctype.startswith("application/json") and body else body
    return status, payload, ms


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):
        return None   # report the 307 itself instead of following it to the shop


def status_no_redirect(url, timeout=30):
    opener = urllib.request.build_opener(_NoRedirect)
    try:
        with opener.open(url, timeout=timeout) as r:
            return r.status, r.headers.get("Location")
    except urllib.error.HTTPError as e:
        return e.code, e.headers.get("Location")


def post_json(url, obj, timeout=180):
    return _req(url, "POST", json.dumps(obj).encode(), {"Content-Type": "application/json"}, timeout)


def post_audio(url, wav_bytes, fields):
    boundary = uuid.uuid4().hex
    parts = []
    for k, v in fields.items():
        parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{k}"\r\n\r\n{v}\r\n'.encode())
    parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="audio"; filename="q.wav"\r\n'
                 f"Content-Type: audio/wav\r\n\r\n".encode() + wav_bytes + b"\r\n")
    parts.append(f"--{boundary}--\r\n".encode())
    return _req(url, "POST", b"".join(parts), {"Content-Type": f"multipart/form-data; boundary={boundary}"})


def tone_wav(seconds=1.5, sr=16000):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1), w.setsampwidth(2), w.setframerate(sr)
        w.writeframes(b"".join(struct.pack("<h", int(3000 * math.sin(2 * math.pi * 220 * i / sr)))
                               for i in range(int(seconds * sr))))
    return buf.getvalue()


def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok)))
    print(f"{'✅' if ok else '❌'} {name:22s} {detail}")
    return ok


def pct(values, p):
    v = sorted(values)
    k = (len(v) - 1) * p / 100
    f, c = math.floor(k), math.ceil(k)
    return v[f] if f == c else v[f] + (v[c] - v[f]) * (k - f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", default="http://127.0.0.1:8000")
    ap.add_argument("--wait", type=int, default=1200, help="seconds to wait for /health = ok")
    ap.add_argument("--latency", type=int, default=0, help="extra sequential /recommend calls for P50/P95")
    ap.add_argument("--audio", help="a real recording (wav/m4a/mp3) for the voice checks")
    a = ap.parse_args()
    base = a.url.rstrip("/")

    # 1) health — models load in the background after the container starts
    t0, health = time.time(), {}
    while time.time() - t0 < a.wait:
        try:
            _, health, _ = _req(base + "/health", timeout=10)
            if isinstance(health, dict) and health.get("status") in ("ok", "error"):
                break
        except Exception:
            pass
        print(f"   … waiting for models ({time.time() - t0:.0f}s) {health.get('status') if isinstance(health, dict) else ''}")
        time.sleep(15)
    if not check("health", isinstance(health, dict) and health.get("status") == "ok",
                 json.dumps({k: health.get(k) for k in ("llm", "asr", "tts", "index_size", "error")} if isinstance(health, dict) else health)):
        sys.exit(1)

    # 2) text recommendation + follow-up
    s, r, ms = post_json(base + "/recommend", {"query": "white sneakers", "session_id": SESSION})
    ok = s == 200 and isinstance(r, dict) and r.get("products")
    check("recommend", ok, f"{ms:,.0f} ms, {len(r.get('products', [])) if isinstance(r, dict) else 0} products")
    if ok:
        s2, r2, ms2 = post_json(base + "/recommend", {"query": "only under $50", "session_id": SESSION,
                                                       "history": [r["history_entry"]]})
        check("follow-up", s2 == 200, f"{ms2:,.0f} ms, parsed={r2.get('parsed', {}) if isinstance(r2, dict) else r2}")
        s3, _, _ = post_json(base + "/feedback", {"request_id": r["request_id"], "rating": "up", "session_id": SESSION,
                                                  "comment": "phase11 smoke test"})
        check("feedback", s3 == 200)
        pid = r["products"][0]["id"]
        s4, loc = status_no_redirect(f"{base}/click/{r['request_id']}/{pid}?rank=1&session_id={SESSION}")
        check("click", s4 in (302, 303, 307), f"http {s4} → {loc}")

    # 3) voice (Phase 10)
    wav = open(a.audio, "rb").read() if a.audio else tone_wav()
    s, t, ms = post_audio(base + "/transcribe", wav, {"session_id": SESSION})
    check("transcribe", s == 200, f"{ms:,.0f} ms, heard: {t.get('text')!r}" if isinstance(t, dict) else f"http {s}")
    s, v, ms = post_audio(base + "/recommend/audio", wav, {"session_id": SESSION})
    # a tone has no words: 422 "No speech" is the correct answer for it
    check("recommend/audio", s == 200 or (s == 422 and not a.audio), f"http {s}, {ms:,.0f} ms")

    # 4) read-aloud (only if the server enabled it)
    if health.get("tts"):
        s, w, ms = post_json(base + "/speak", {"text": "Here are two white sneakers under $50."})
        check("speak", s == 200 and isinstance(w, (bytes, bytearray)) and w[:4] == b"RIFF", f"{ms:,.0f} ms")

    s, m, _ = _req(base + "/metrics", timeout=10)
    check("metrics", s == 200, json.dumps(m.get("counts")) if isinstance(m, dict) else "")

    # 5) optional latency sample
    if a.latency:
        qs = ["white sneakers", "a gray sofa", "black leather wallet", "gold hoop earrings for women", "black tote bag"]
        times = []
        for i in range(a.latency):
            s, r, ms = post_json(base + "/recommend", {"query": qs[i % len(qs)], "session_id": SESSION})
            if s == 200:
                times.append(ms)
        if times:
            print(f"   latency over {len(times)} requests: P50 {pct(times, 50):,.0f} ms · P95 {pct(times, 95):,.0f} ms"
                  f" · mean {statistics.mean(times):,.0f} ms")
        check("latency sample", len(times) == a.latency, f"{len(times)}/{a.latency} ok")

    failed = [n for n, ok in RESULTS if not ok]
    print("\nALL CHECKS PASSED" if not failed else f"\nFAILED: {failed}")
    sys.exit(1 if failed else 0)


if __name__ == "__main__":
    main()

### 📘 2.5 `deploy/ec2.sh`

**Purpose:** on the EC2 instance: `setup` (download artifacts, build), `start`, `smoke`, `status`, `logs`, `tunnel`, `save-logs`, `stop`

**Expected Results:** `Writing /content/ShopTalk/deploy/ec2.sh`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/deploy/ec2.sh
#!/usr/bin/env bash
# ShopTalk on one EC2 GPU instance (g4dn.xlarge, "Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04)").
# Run from the cloned repo on the instance:
#
#   ./deploy/ec2.sh setup "<presigned S3 URL of shoptalk_artifacts.zip>"   # once: models + index, build image
#   ./deploy/ec2.sh start      # run the container (first start downloads ~8 GB of models: 5–10 min)
#   ./deploy/ec2.sh smoke      # end-to-end checks + a 20-request latency sample
#   ./deploy/ec2.sh status     # container, GPU memory, /health
#   ./deploy/ec2.sh logs       # follow the container output (Ctrl+C to leave)
#   ./deploy/ec2.sh tunnel     # optional public https link (Cloudflare quick tunnel) instead of an SSH tunnel
#   ./deploy/ec2.sh save-logs  # pack requests/feedback logs into ~/shoptalk/shoptalk_logs_<time>.tgz
#   ./deploy/ec2.sh stop       # stop the container — then STOP THE INSTANCE in the AWS console
set -euo pipefail

REPO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
BASE="${SHOPTALK_HOME:-$HOME/shoptalk}"          # artifacts, model cache and logs live here (survive rebuilds)
IMAGE="${SHOPTALK_IMAGE:-shoptalk:gpu}"
NAME=shoptalk
DOCKER=docker
docker ps >/dev/null 2>&1 || DOCKER="sudo docker"

say() { printf '\n\033[1m%s\033[0m\n' "$*"; }
die() { printf '\n❌ %s\n' "$*" >&2; exit 1; }

preflight() {
  command -v nvidia-smi >/dev/null || die "nvidia-smi not found — use the 'Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04)'"
  nvidia-smi -L || die "no GPU visible — is this a g4dn instance?"
  $DOCKER info >/dev/null 2>&1 || die "Docker is not running"
  $DOCKER info 2>/dev/null | grep -qi nvidia || echo "⚠️ NVIDIA runtime not listed by 'docker info' — '--gpus all' may still work via CDI"
}

cmd_setup() {
  local url="${1:-}"
  [[ -n "$url" ]] || die "usage: $0 setup \"<presigned URL of shoptalk_artifacts.zip>\"  (keep the quotes)"
  preflight
  mkdir -p "$BASE"/{artifacts,hf-cache,logs}
  say "1/3 Downloading models + Chroma index"
  curl -fL --retry 3 -o "$BASE/artifacts.zip" "$url" || die "download failed — the presigned URL may have expired (make a new one in S3)"
  python3 -m zipfile -e "$BASE/artifacts.zip" "$BASE/artifacts/"
  [[ -f "$BASE/artifacts/data/chroma/chroma.sqlite3" ]] || die "artifacts.zip has no data/chroma/chroma.sqlite3"
  ls -d "$BASE"/artifacts/models/*/ >/dev/null 2>&1 || die "artifacts.zip has no models/<embedding model>/"
  du -sh "$BASE/artifacts"/*
  say "2/3 Building the image (GPU build of torch, ~5–8 min the first time)"
  $DOCKER build -t "$IMAGE" "$REPO_DIR"
  say "3/3 Done. Next: $0 start"
}

cmd_start() {
  [[ -d "$BASE/artifacts/data/chroma" ]] || die "run '$0 setup <url>' first"
  $DOCKER rm -f "$NAME" >/dev/null 2>&1 || true
  # Ports are bound to 127.0.0.1 only: reach them through an SSH tunnel (or '$0 tunnel'), never an open port.
  $DOCKER run -d --name "$NAME" --gpus all --restart unless-stopped --shm-size 2g \
    -p 127.0.0.1:7860:7860 -p 127.0.0.1:8000:8000 \
    -v "$BASE/artifacts/models:/app/models:ro" \
    -v "$BASE/artifacts/data:/app/data:ro" \
    -v "$BASE/hf-cache:/cache/huggingface" \
    -v "$BASE/logs:/app/logs" \
    -e SHOPTALK_TTS="${SHOPTALK_TTS:-1}" \
    "$IMAGE" >/dev/null
  say "Container started. Models load in the background (first start downloads ~8 GB: 5–10 min)."
  echo "Watch: $0 logs     ·  then: $0 smoke"
}

cmd_smoke()  { $DOCKER exec "$NAME" python /app/docker/smoke_test.py --latency "${LATENCY_N:-20}" "$@"; }
cmd_logs()   { $DOCKER logs -f --tail 100 "$NAME"; }
cmd_status() {
  $DOCKER ps --filter "name=$NAME" --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'
  nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
  curl -s --max-time 5 http://127.0.0.1:8000/health && echo || echo "(API not answering yet)"
}
cmd_tunnel() {
  local exe="$BASE/cloudflared"
  [[ -x "$exe" ]] || { curl -fsSL -o "$exe" https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x "$exe"; }
  say "Public https link for the UI below (Ctrl+C closes it). Anyone with the link can use ShopTalk — close it after the demo."
  "$exe" tunnel --url http://127.0.0.1:7860 --protocol http2 --no-autoupdate
}
cmd_save_logs() {
  local f
  f="$BASE/shoptalk_logs_$(date +%Y%m%d-%H%M).tgz"
  tar -czf "$f" -C "$BASE" logs && echo "saved $f  (download: scp -i <key.pem> ubuntu@<ip>:$f .)"
}
cmd_stop() {
  $DOCKER stop "$NAME" >/dev/null 2>&1 && echo "container stopped" || echo "container was not running"
  say "Now STOP the instance: AWS console ▸ EC2 ▸ Instances ▸ select ▸ Instance state ▸ Stop (GPU billing ends; the disk is kept)."
}

case "${1:-}" in
  setup) shift; cmd_setup "$@" ;;
  start) cmd_start ;;
  smoke) shift; cmd_smoke "$@" ;;
  logs) cmd_logs ;;
  status) cmd_status ;;
  tunnel) cmd_tunnel ;;
  save-logs) cmd_save_logs ;;
  stop) cmd_stop ;;
  *) sed -n '2,13p' "$0"; exit 1 ;;
esac

### 📘 2.6 `deploy/README.md`

**Purpose:** the step-by-step runbook: quota, local test, S3, EC2 launch, SSH tunnel, demo shot list, shutdown, troubleshooting

**Expected Results:** `Writing /content/ShopTalk/deploy/README.md`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/deploy/README.md
# ShopTalk — Phase 11: Deployment runbook

One Docker image runs the API (FastAPI, port 8000) and the UI (Gradio, port 7860). You test it on your laptop in
**fake mode** (no GPU needed), then run the real thing on **one EC2 GPU instance** for the demo and stop it right after.

| Step | Where | Time | Cost |
|---|---|---|---|
| 1. GPU quota + budget alert | AWS console | 10 min (+ approval wait, often 1–2 days) | free |
| 2. Prepare repo + artifacts | Colab: `ShopTalk_Phase11_Prepare.ipynb` | 15 min | free |
| 3. Local Docker test (fake mode) | your laptop | 30–60 min (first build) | free |
| 4. Deploy on EC2 | AWS + PowerShell | 45–60 min | g4dn.xlarge ≈ $0.53/h |
| 5. Record the demo | browser on your laptop | 30–60 min | (instance running) |
| 6. Stop / tear down | AWS console | 5 min | disk ≈ $0.27/day until terminated |

**Why EC2 g4dn.xlarge:** same Tesla T4 as Colab, so the Phase 9–10 latency numbers carry over; the
"Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04)" already has the NVIDIA driver, Docker and the NVIDIA
Container Toolkit. SageMaker endpoints expect one model behind `/invocations`, not an API + UI pair, and cost more
per hour; a CPU instance would take tens of seconds per answer with Qwen-3B + Whisper.

---

## 1. Today: GPU quota and a budget alert

**GPU quota** (new accounts start at 0 for GPU instances):
1. AWS console → choose your region (e.g. **US East (N. Virginia) us-east-1**) top right.
2. **Service Quotas → AWS services → Amazon Elastic Compute Cloud (Amazon EC2)**.
3. Search **Running On-Demand G and VT instances** → **Request increase at account level** → **8** (it counts vCPUs;
   g4dn.xlarge uses 4). Reason: "Course project demo: one g4dn.xlarge for a few hours."
4. Wait for the approval email. Don't confuse it with "All G and VT **Spot** Instance Requests" — that is a different quota.

**Budget alert** (so a forgotten instance can't surprise you):
**Billing and Cost Management → Budgets → Create budget → Use a template → Monthly cost budget**, amount **$20**,
your email. You get an email at 85% and 100%.

---

## 2. Prepare the repo and the artifacts (Colab)

Run `ShopTalk_Phase11_Prepare.ipynb` top to bottom. It:
- writes `Dockerfile`, `.dockerignore`, `docker/start.sh`, `docker/smoke_test.py`, `deploy/ec2.sh` and this guide into the repo on Drive;
- writes **`requirements-docker.txt`**: the exact package versions from the Colab runtime the project was tested on;
- zips the two things the container needs but GitHub shouldn't hold — the LoRA embedding model and the Chroma index —
  into **`deploy_artifacts/shoptalk_artifacts.zip`** on Drive;
- pushes the current Drive code (Phases 7–10 included) to **GitHub** (needs a personal access token; see the notebook).

Then download `shoptalk_artifacts.zip` from Drive to your laptop (you upload it to S3 in step 4).

---

## 3. Local test on your laptop (fake mode)

Fake mode (`SHOPTALK_FAKE=1`) runs the same image, endpoints, UI, microphone and feedback buttons with stand-in models,
so it works without an NVIDIA GPU. It proves the container is correct; the real models are tested on EC2.

**Install Docker Desktop (Windows):**
1. Download **Docker Desktop for Windows** from docs.docker.com/desktop → install with **"Use WSL 2"** ticked → restart.
2. Start Docker Desktop and wait for "Engine running". In PowerShell: `docker run --rm hello-world` → "Hello from Docker!".
   If it asks for WSL: `wsl --install` in an administrator PowerShell, restart, try again.

**Build and run** (PowerShell, in the cloned repo folder: `git clone https://github.com/<you>/<repo>.git; cd <repo>`):
```powershell
docker build --build-arg TORCH_INDEX=https://download.pytorch.org/whl/cpu -t shoptalk:cpu .
docker run --rm -d --name shoptalk -p 7860:7860 -p 8000:8000 -e SHOPTALK_FAKE=1 -e SHOPTALK_TTS=1 shoptalk:cpu
docker exec shoptalk python /app/docker/smoke_test.py
```
Expected: `ALL CHECKS PASSED` (answers are fake: "red sneakers", "Show me a red shirt…").
Then open **http://localhost:7860** in your browser: type a request, press 🎤 (localhost is allowed to use the
microphone), try 👍/👎 and "View product". Stop with `docker stop shoptalk`.

The CPU build is ~1.5 GB; the first build takes 5–15 min depending on your internet.

---

## 4. Deploy on EC2

### 4.1 Upload the artifacts to S3
1. **S3 → Create bucket** → name e.g. `shoptalk-artifacts-<yourname>` (same region), keep "Block all public access" ON.
2. Open the bucket → **Upload** → `shoptalk_artifacts.zip`.
3. Click the uploaded file → **Object actions → Share with a presigned URL** → **12 hours** → **Create** → copy the URL.
   (The URL lets the instance download this one file without AWS keys; it expires by itself.)

### 4.2 Launch the instance
**EC2 → Launch instance**:

| Setting | Value |
|---|---|
| Name | `shoptalk-demo` |
| AMI | search **Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04)** (x86_64) |
| Instance type | **g4dn.xlarge** |
| Key pair | **Create new key pair** → `shoptalk-key`, RSA, **.pem** → it downloads; keep it |
| Network | Allow SSH traffic from **My IP** (nothing else — the UI is reached through the SSH tunnel) |
| Storage | **100 GiB gp3** |
| Purchasing | On-Demand (Spot is ~50% cheaper but can be interrupted mid-demo) |

**Launch**, wait until "Running" with 2/2 status checks, and copy the **Public IPv4 address**.

### 4.3 Connect from Windows (PowerShell)
Fix the key's permissions once (Windows SSH refuses keys others can read):
```powershell
cd $HOME\Downloads
icacls.exe shoptalk-key.pem /reset
icacls.exe shoptalk-key.pem /grant:r "$($env:USERNAME):(R)"
icacls.exe shoptalk-key.pem /inheritance:r
```
Connect **with the tunnel** (your laptop's ports 7860/8000 now lead to the instance):
```powershell
ssh -i shoptalk-key.pem -L 7860:localhost:7860 -L 8000:localhost:8000 ubuntu@<PUBLIC-IP>
```
Type `yes` the first time. Keep this window open while you work and record.

### 4.4 Set up and start (on the instance)
```bash
git clone https://github.com/<you>/<repo>.git shoptalk && cd shoptalk
#   private repo: git clone https://<github-token>@github.com/<you>/<repo>.git shoptalk
bash deploy/ec2.sh setup "<PRESIGNED-URL>"     # keep the quotes; ~10 min (download + GPU build)
bash deploy/ec2.sh start
bash deploy/ec2.sh logs                        # wait for the models (first start ~5–10 min), Ctrl+C to leave
bash deploy/ec2.sh smoke                       # ALL CHECKS PASSED + a latency sample
```
Now open **http://localhost:7860** on your laptop (through the tunnel). The status line should show
🟢 API ready · Qwen · 🎤 whisper.

Need a link someone else can open? `bash deploy/ec2.sh tunnel` prints a public https URL (Ctrl+C closes it).

---

## 5. Record the demo (plan item 11.3)

**Before recording:** run `bash deploy/ec2.sh smoke` once (warms the models) and try your exact demo queries — pick
ones that work. Close other tabs; zoom the browser to 110–125%.

**Tools (Windows):** Xbox Game Bar (**Win + Alt + R** records the active window with your microphone) or OBS Studio
for the full screen; trim in Clipchamp. Aim for **3–5 minutes**.

**Shot list** (covers everything the plan asks for):
| # | Show | Example |
|---|---|---|
| 1 | Intro: what ShopTalk is, running on AWS (status line, 5 s) | — |
| 2 | Simple product query | "white sneakers" |
| 3 | Query with constraints (price/colour) | "a blue area rug under $100" |
| 4 | **Follow-up** in the same chat | "show me cheaper ones" or "in gray" |
| 5 | Another category | "gold hoop earrings for women" / "a gray sofa" |
| 6 | Something the catalog doesn't have → honest "no match" | test one first, e.g. "a used car" |
| 7 | **Spoken query** with 🎤 — point at the transcript in the box, then Send | "black leather shoes for men" |
| 8 | Optional: 🔊 read aloud | — |
| 9 | **Feedback**: 👍 on a good answer, 👎 with a comment on another, click **View product** | — |
| 10 | Proof it was logged (terminal) | `tail -n 3 ~/shoptalk/logs/feedback.jsonl` |

Avoid the known Phase 9 failure queries (sun hat, gemstone ring, diamond studs) unless you show one on purpose as a
known limitation with the fix from `failure_cases_final.md`.

---

## 6. Stop and tear down

**Right after recording:**
```bash
bash deploy/ec2.sh save-logs     # optional: keep the demo's request/feedback logs
bash deploy/ec2.sh stop
```
Download the logs if you want them (PowerShell, new window):
`scp -i shoptalk-key.pem ubuntu@<PUBLIC-IP>:~/shoptalk/shoptalk_logs_*.tgz .`

Then **EC2 → Instances → shoptalk-demo → Instance state → Stop.** GPU billing ends; the 100 GB disk stays
(≈ $8/month, ≈ $0.27/day) so you can start it again for a retake (the public IP changes; models are already cached).

**When the project is graded:**
1. **Terminate** the instance (Instance state → Terminate) — this deletes its disk too.
2. **EC2 → Volumes / Snapshots**: nothing left with your instance's name.
3. **S3**: delete `shoptalk_artifacts.zip` and the bucket.
4. **Billing → Bills** the next day: no running EC2 charges.

---

## 7. Troubleshooting

| Symptom | Fix |
|---|---|
| Launch error *VcpuLimitExceeded* / *You have requested more vCPU capacity…* | GPU quota not approved yet (step 1), or requested in another region |
| *InsufficientInstanceCapacity* | No T4s free in that zone: retry later, or pick another subnet/AZ in the launch "Network settings" |
| `ssh: … UNPROTECTED PRIVATE KEY FILE` | Run the three `icacls` commands in 4.3 |
| `ssh: connection timed out` | Security group SSH rule is for an old IP: EC2 → Security groups → edit inbound → My IP |
| `setup` download fails | Presigned URL expired (12 h): create a new one (4.1 step 3) |
| `/health` stays `loading` | First start downloads ~8 GB from Hugging Face: `bash deploy/ec2.sh logs` |
| `CUDA error: no kernel image` / torch can't see GPU | Use the Ubuntu 24.04 Base OSS DLAMI (driver supports CUDA 13); `nvidia-smi` must work on the host |
| UI loads but 🎤 says no microphone | Use `http://localhost:7860` through the SSH tunnel (or the https `tunnel` link) — not `http://<public-ip>` |
| Port 7860 already in use on the laptop | Stop the local container: `docker stop shoptalk` |
| Disk full | `docker system prune` (old images), or 100 GiB storage at launch |

### 📘 2.7 Test the container's start script here (fake mode, no Docker)

**Purpose:** Colab cannot run Docker, but it can run exactly what the container runs: `docker/start.sh` with the repo's current `src/`, in fake mode, followed by `docker/smoke_test.py`. This catches wiring errors before the laptop build.

**Expected Results:** `ALL CHECKS PASSED`; `entrypoint_fake_test` = PASS.

**Parameters:** none

In [ ]:
APP = Path("/content/app_test")
shutil.rmtree(APP, ignore_errors=True); APP.mkdir()
shutil.copytree(REPO_LINK / "src", APP / "src", ignore=shutil.ignore_patterns("__pycache__", "_backup_before_phase10"))
shutil.copytree(REPO_LINK / "docker", APP / "docker")
env = {**os.environ, "APP_DIR": str(APP), "PYTHONPATH": str(APP), "SHOPTALK_FAKE": "1", "SHOPTALK_TTS": "1",
       "SHOPTALK_REPO_DIR": str(APP), "SHOPTALK_LOG_DIR": str(APP / "logs"), "SHOPTALK_API_PORT": "8030",
       "SHOPTALK_API_URL": "http://127.0.0.1:8030", "SHOPTALK_UI_PORT": "7890"}
proc = subprocess.Popen(["bash", str(APP / "docker/start.sh")], env=env, stdout=open(APP / "start.log", "w"),
                        stderr=subprocess.STDOUT, start_new_session=True)
try:
    r = subprocess.run([sys.executable, str(APP / "docker/smoke_test.py"), "--url", "http://127.0.0.1:8030", "--wait", "120"],
                       capture_output=True, text=True, timeout=300)
    print(r.stdout[-3000:], r.stderr[-1500:])
    ok = r.returncode == 0
finally:
    os.killpg(proc.pid, 15); proc.wait(timeout=30)
print((APP / "start.log").read_text()[-800:])
check("entrypoint_fake_test", "PASS" if ok else "FAIL", "start.sh + smoke_test in fake mode")

## 3. Exact package versions for the image

### 📘 3.1 Write `requirements-docker.txt`

**Purpose:** Pin every package the code needs — and everything those packages need — to the version installed in this Colab runtime (the environment Phases 7–10 were tested in). torch and the NVIDIA libraries are left out: the Dockerfile installs torch separately from the CUDA or CPU index.

**Expected Results:** `requirements-docker.txt` with ~80–150 pins; `requirements_docker` = PASS.

**Parameters:** none

In [ ]:
from importlib.metadata import distribution, packages_distributions
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name as canon

EXCLUDE = re.compile(r"^(torch|torchvision|torchaudio|triton|nvidia-.*|cuda-.*|pip|setuptools|wheel)$")
def top_level():
    reqs = []
    for name in ("requirements-api.txt", "requirements-ui.txt"):
        p = REPO_LINK / name
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.split("#")[0].strip()
                if line and not line.startswith("-"):
                    reqs.append(Requirement(line))
    # packages the code imports directly, in case a requirements file left one out
    for extra in ["transformers", "sentence-transformers", "peft", "chromadb", "fastapi", "uvicorn", "python-multipart",
                  "pydantic", "requests", "gradio", "numpy", "pandas", "psutil", "accelerate"]:
        reqs.append(Requirement(extra))
    return reqs

PINS, VISITED = {}, {}
def walk(name, extras=()):
    key = canon(name)
    if EXCLUDE.match(key): return
    try: dist = distribution(name)
    except Exception: return                       # optional dependency that isn't installed here
    seen = VISITED.setdefault(key, set())
    if key in PINS and set(extras) <= seen: return
    PINS[key] = dist.version
    seen |= set(extras)
    for r in (dist.requires or []):
        req = Requirement(r)
        # keep a dependency if its marker holds here, either plainly or for one of the extras we asked for
        if req.marker and not any(req.marker.evaluate({"extra": e}) for e in ("",) + tuple(extras)):
            continue
        walk(req.name, tuple(req.extras))
for r in top_level():
    walk(r.name, tuple(r.extras))
lines = [f"{k}=={v}" for k, v in sorted(PINS.items())]
try: PINS_TORCH = version("torch")
except PackageNotFoundError: PINS_TORCH = "not installed"
header = [f"# Generated by ShopTalk_Phase11_Prepare.ipynb on {time.strftime('%Y-%m-%d')} from the Colab runtime",
          f"# Python {sys.version.split()[0]} · torch {PINS_TORCH} (installed separately in the Dockerfile)"]
(REPO_LINK / "requirements-docker.txt").write_text("\n".join(header + lines) + "\n")
print(len(lines), "pins →", REPO_LINK / "requirements-docker.txt")
print("\n".join(l for l in lines if l.split("==")[0] in ("transformers", "sentence-transformers", "peft", "chromadb", "torchao", "gradio", "fastapi", "python-multipart")))

# every third-party module imported by src/ must be covered
import ast, sysconfig
stdlib = set(sys.stdlib_module_names)
mods = set()
for f in (REPO_LINK / "src").rglob("*.py"):
    if "_backup" in str(f): continue
    for node in ast.walk(ast.parse(f.read_text(errors="ignore"))):
        if isinstance(node, ast.Import): mods |= {a.name.split(".")[0] for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module and not node.level: mods.add(node.module.split(".")[0])
pd_map = packages_distributions()
missing = []
NOTEBOOK_ONLY = {"src", "google", "IPython", "torch", "ipywidgets", "tqdm"}   # used by the Colab notebooks, not the server
for m in sorted(mods - stdlib - NOTEBOOK_ONLY):
    dists = {canon(d) for d in pd_map.get(m, [])}
    if dists and not dists & set(PINS): missing.append(f"{m} ({', '.join(dists)})")
    elif not dists: missing.append(f"{m} (not installed here)")
check("requirements_docker", "PASS" if not missing else "WARN",
      f"{len(lines)} pins" + (f"; not covered: {missing}" if missing else "; every import in src/ is covered"))

## 4. Artifacts for S3

### 📘 4.1 Zip the embedding model and the Chroma index

**Purpose:** These two folders are needed at run time but don't belong in GitHub. The zip keeps the repo-relative paths (`models/…`, `data/chroma/…`), which `deploy/ec2.sh setup` unpacks next to the container.

**Expected Results:** `deploy_artifacts/shoptalk_artifacts.zip` on Drive with its size and SHA-256; `artifacts_zip` = PASS.

**Parameters:** none

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
tmp = Path("/content/shoptalk_artifacts.zip")
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root in (EMBED_DIR, CHROMA_DIR):
        for f in root.rglob("*"):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(REPO_LINK).as_posix())
with zipfile.ZipFile(tmp) as z:
    names = z.namelist()
    assert any(n.endswith("chroma.sqlite3") for n in names), "no chroma.sqlite3 in the zip"
sha = hashlib.sha256(tmp.read_bytes()).hexdigest()
shutil.copy2(tmp, ARTIFACTS_DIR / tmp.name)
(ARTIFACTS_DIR / "shoptalk_artifacts.sha256").write_text(f"{sha}  {tmp.name}\n")
mb = tmp.stat().st_size / 1e6
print(f"{len(names)} files, {mb:.0f} MB, sha256 {sha[:16]}…  →  {ARTIFACTS_DIR / tmp.name}")
print("top-level:", sorted({'/'.join(n.split('/')[:2]) for n in names}))
check("artifacts_zip", "PASS", f"{mb:.0f} MB with {RAG_CFG['embed_model']} + {RAG_CFG['chroma_dir']}")

## 5. Push to GitHub

### 📘 5.1 Sync the Drive code to GitHub

**Purpose:** Clone your repo, copy in `src/` (Phases 7–10, without backups or caches), the deployment files, the requirements files and the phase notebooks, extend `.gitignore`, and show the change summary. With `PUSH = True` it commits and pushes. It needs a GitHub **personal access token** (GitHub → Settings → Developer settings → Personal access tokens → Fine-grained → this repo → Contents: Read and write); the token is typed into a hidden prompt and not saved.

**Expected Results:** `git status` + `diff --stat`; with `PUSH = True`: `github_push` = PASS and the commit hash.

**Parameters:** `GITHUB_REPO`, `GIT_BRANCH`, `PUSH`, `NOTEBOOK_GLOBS`.

In [ ]:
from getpass import getpass
assert "<your-user>" not in GITHUB_REPO, "set GITHUB_REPO in 1.2"
token = getpass("GitHub personal access token (hidden): ").strip()
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
CLONE = Path("/content/gh_shoptalk")
shutil.rmtree(CLONE, ignore_errors=True)
def git(*args, check_ok=True):
    r = subprocess.run(["git", "-C", str(CLONE), *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check_ok and r.returncode: raise RuntimeError(out)
    return out
r = subprocess.run(["git", "clone", "--branch", GIT_BRANCH, auth_url, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, (r.stdout + r.stderr).replace(token, "***")
git("config", "user.name", GIT_USER_NAME); git("config", "user.email", GIT_USER_EMAIL)

ignore = shutil.ignore_patterns("__pycache__", "*.pyc", "_backup_before_phase10", ".ipynb_checkpoints")
for d in SYNC_DIRS:
    if (REPO_LINK / d).exists():
        shutil.rmtree(CLONE / d, ignore_errors=True)          # mirror: files deleted on Drive disappear here too
        shutil.copytree(REPO_LINK / d, CLONE / d, ignore=ignore)
for f in SYNC_FILES:
    if (REPO_LINK / f).exists(): shutil.copy2(REPO_LINK / f, CLONE / f)
nbs = sorted({p for g in NOTEBOOK_GLOBS for p in glob.glob(g)})
(CLONE / "notebooks").mkdir(exist_ok=True)
for p in nbs: shutil.copy2(p, CLONE / "notebooks" / Path(p).name)
print("notebooks copied:", [Path(p).name for p in nbs] or "none found — check NOTEBOOK_GLOBS")
gi = CLONE / ".gitignore"
have = gi.read_text().splitlines() if gi.exists() else []
gi.write_text("\n".join(have + [l for l in GITIGNORE if l not in have]) + "\n")
git("add", "-A")
for f in ("docker/start.sh", "deploy/ec2.sh"):
    git("update-index", "--chmod=+x", f, check_ok=False)
print(git("status", "--short")[:4000])
print(git("diff", "--cached", "--stat")[-2000:])
big = [l for l in git("diff", "--cached", "--numstat").splitlines() if l.startswith("-")]
if PUSH:
    git("commit", "-m", "Phases 7-11: API, UI, evaluation, voice input, Docker + EC2 deployment files")
    print(git("push", "origin", GIT_BRANCH))
    commit = git("rev-parse", "--short", "HEAD").strip()
    check("github_push", "PASS", f"pushed {commit} to {GIT_BRANCH}")
else:
    check("github_push", "PENDING", "review the changes above, then set PUSH = True in 1.2 and re-run 1.2 + this cell")
git("remote", "set-url", "origin", GITHUB_REPO)                 # don't leave the token in the clone's config
del token

## 6. Checklist and next steps

### 📘 6.1 Prep checklist

**Purpose:** Summary of this notebook and what to do next (from `deploy/README.md`).

**Expected Results:** All PASS (github_push PASS after the push run).

**Parameters:** none

In [ ]:
order = ["repo_ready", "entrypoint_fake_test", "requirements_docker", "artifacts_zip", "github_push"]
display(pd.DataFrame([{"check": k, **CHECKS.get(k, {"status": "MISSING", "detail": "cell not run"})} for k in order]))
print('''
Next (details in deploy/README.md):
  1. TODAY  AWS Service Quotas → "Running On-Demand G and VT instances" → 8 (your region) + a $20 budget alert
  2. Laptop Docker Desktop → git clone → docker build (CPU) → docker run (fake) → smoke test → try http://localhost:7860
  3. S3     upload deploy_artifacts/shoptalk_artifacts.zip (download it from Drive first) → presigned URL (12 h)
  4. EC2    g4dn.xlarge + Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04), 100 GiB, SSH from My IP
            ssh -L 7860:localhost:7860 … → bash deploy/ec2.sh setup "<url>" → start → smoke → open http://localhost:7860
  5. Demo   follow the shot list, then:  bash deploy/ec2.sh stop  →  EC2 ▸ Stop instance
''')